# 03 - Silver Layer: Data Cleansing & Validation
Apply validation rules, standardize schemas, persist managed Delta tables, and export Silver Parquet snapshots.

In [0]:
%run ./00_setup_config

# 00 - Setup & Configuration
This notebook sets up schemas, paths, and runtime variables for the Food Inspection project.

Medallion layers are separated by schema for Delta tables and by folder path for Parquet snapshots.

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


All widgets cleared.


## Widget Parameters

raw_path    = /Volumes/workspace/food_inspection/raw_data
bronze_path = /Volumes/workspace/foodinspection_bronze/food_inspection
silver_path = /Volumes/workspace/foodinspection_silver/food_inspection
gold_path   = /Volumes/workspace/foodinspection_gold/food_inspection
bronze_schema = foodinspection_bronze
silver_schema = foodinspection_silver
gold_schema   = foodinspection_gold


...

Schemas ready: foodinspection_bronze, foodinspection_silver, foodinspection_gold


## Verify Raw Data Files Exist

Raw data files:
  Food_Inspections_20260411.csv (330.70 MB)
  Restaurant_and_Food_Establishment_Inspections_(October_2016_to_January_2024)_20260411.csv (192.79 MB)
  backup_Food_Inspections_20260411.csv (330.70 MB)
  parquet_snapshots/ (0.00 MB)


## Helper Utilities

Configuration ready. Variables and snapshot helpers are available via %run ./00_setup_config


In [0]:
from pyspark.sql.functions import (
    col, trim, upper, when, lit, regexp_replace, regexp_extract,
    to_date, split, explode, array, struct, expr,
    monotonically_increasing_id, concat_ws, coalesce, length,
    sum as spark_sum, current_timestamp, sha2
)
from pyspark.sql.types import IntegerType


def add_silver_lineage(df, key_columns):
    """Add Silver layer lineage columns: updated_at and durability_key."""
    return (
        df
        .withColumn("updated_at", current_timestamp())
        .withColumn(
            "durability_key",
            sha2(
                concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in key_columns]),
                256,
            ),
        )
    )

## 1. Load Bronze Tables

In [0]:
df_chicago_bronze = spark.table(f"{BRONZE_SCHEMA}.chicago_inspections")
df_dallas_bronze = spark.table(f"{BRONZE_SCHEMA}.dallas_inspections")

print(f"Chicago Bronze count: {df_chicago_bronze.count()}")
print(f"Dallas Bronze count: {df_dallas_bronze.count()}")

Chicago Bronze count: 308431
Dallas Bronze count: 78984


## 2. Chicago Silver Cleansing

In [0]:
df_chicago_clean = (
    df_chicago_bronze
    .withColumn("DBA_Name", trim(regexp_replace(col("DBA_Name"), '"', '')))
    .withColumn("AKA_Name", trim(regexp_replace(col("AKA_Name"), '"', '')))
    .withColumn("Address", trim(col("Address")))
    .withColumn("City", trim(upper(col("City"))))
    .withColumn("State", trim(upper(col("State"))))
    .withColumn("Results", trim(col("Results")))
    .withColumn("Inspection_Type", trim(col("Inspection_Type")))
    .withColumn("Facility_Type", trim(col("Facility_Type")))
    .withColumn("Risk", trim(col("Risk")))
    .withColumn("Zip", regexp_replace(col("Zip").cast("string"), "\\.0$", ""))
    .withColumn("Zip", when(length(col("Zip")) == 4, concat_ws("", lit("0"), col("Zip"))).otherwise(col("Zip")))
    .withColumn("Inspection_Date", to_date(col("Inspection_Date"), "MM/dd/yyyy"))
    .withColumn("Latitude", expr("try_cast(Latitude as double)"))
    .withColumn("Longitude", expr("try_cast(Longitude as double)"))
    .withColumn("source_city", lit("Chicago"))
)

df_chicago_clean = df_chicago_clean.withColumn(
    "Inspection_Score",
    when(col("Results") == "Pass", 90)
    .when(col("Results") == "Pass w/ Conditions", 80)
    .when(col("Results") == "Fail", 70)
    .when(col("Results") == "No Entry", 0)
    .otherwise(lit(None).cast(IntegerType())),
)

chicago_before = df_chicago_clean.count()

df_chicago_clean = df_chicago_clean.filter(col("DBA_Name").isNotNull() & (col("DBA_Name") != ""))
df_chicago_clean = df_chicago_clean.filter(col("Inspection_Date").isNotNull())
df_chicago_clean = df_chicago_clean.filter(col("Inspection_Type").isNotNull() & (col("Inspection_Type") != ""))
df_chicago_clean = df_chicago_clean.filter(col("Zip").isNotNull() & col("Zip").rlike("^\\d{5}$"))
df_chicago_clean = df_chicago_clean.filter(col("Results").isNotNull() & (col("Results") != ""))
df_chicago_clean = df_chicago_clean.filter(col("Violations").isNotNull() & (col("Violations") != ""))
df_chicago_clean = df_chicago_clean.filter(
    ~((upper(col("Results")) == "PASS") & (col("Violations").rlike("(?i)(urgent|critical)")))
)

chicago_after = df_chicago_clean.count()
print(f"Chicago: {chicago_before} -> {chicago_after} rows ({chicago_before - chicago_after} dropped)")

Chicago: 308431 -> 221877 rows (86554 dropped)


In [0]:
df_chicago_violations = (
    df_chicago_clean
    .withColumn("violation_array", split(col("Violations"), "\\|"))
    .withColumn("violation_raw", explode(col("violation_array")))
    .withColumn("violation_raw", trim(col("violation_raw")))
    .filter(col("violation_raw") != "")
    .withColumn("violation_code", regexp_extract(col("violation_raw"), r"^(\d+)\.", 1))
    .withColumn(
        "violation_description",
        trim(regexp_extract(col("violation_raw"), r"^\d+\.\s*(.+?)(?:\s*-\s*Comments:.*)?$", 1)),
    )
    .withColumn("violation_comments", trim(regexp_extract(col("violation_raw"), r"-\s*Comments:\s*(.*)", 1)))
    .withColumn("violation_code", when(col("violation_code") == "", lit(None)).otherwise(col("violation_code")))
    .withColumn(
        "violation_description",
        when(
            col("violation_description") == "",
            trim(regexp_replace(col("violation_raw"), r"^\d+\.\s*", "")),
        ).otherwise(col("violation_description")),
    )
    .withColumn("violation_points", lit(None).cast(IntegerType()))
    .dropDuplicates(["Inspection_ID", "violation_raw"])
)

display(
    df_chicago_violations
    .select("Inspection_ID", "violation_code", "violation_description", "violation_comments")
    .limit(20)
)

Inspection_ID,violation_code,violation_description,violation_comments
2633654,47,"FOOD & NON-FOOD CONTACT SURFACES CLEANABLE, PROPERLY DESIGNED, CONSTRUCTED & USED",OBSERVED RUSTED SHELVING RACKS INSIDE UPRIGHT COCA COLA COOLING EQUIPMENT. INSTRUCTED TO RESURFACE OR REPLACE TO MAINTAIN IN GOOD REPAIR.
2633364,58,ALLERGEN TRAINING AS REQUIRED,NO ALLERGEN CERTIFICATES ON SITE AT TIME OF INSPECTION. INSTRUCTED TO OBTAIN AND MAINTAIN
2633344,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN",OBSERVED THE FLOOR UNDER THE THREE COMPARTMENT SINK AT THE GREASE TRAP IN NEED OF A DETAIL CLEANING...OBSERVED THE FOOD SERVICE AREA FLOOR UNDER AND AROUND ALL COOKING AND NON COOKING EQUIPMENT IN NEED OF A DETAIL CLEANING. INSTRUCTED TO CLEAN AND MAINTAIN AREAS AS A MEANS OF PREVENTING POSSIBLE PEST AND OR INSECT ACTIVITY.
2633332,56,ADEQUATE VENTILATION & LIGHTING; DESIGNATED AREAS USED,"6-501.14 OBSERVED DUST BUILDUP ON FANS, LIGHT SHIELDS IN COOKING AREA AND CEILING AND WALL VENTS IN PREP AND STORAGE AREAS. INSTRUCTED MANAGER TO CLEAN AND MAINTAIN ALL FANS, LIGHT SHIELDS AND CEILING/WALL VENTS."
2633351,58,ALLERGEN TRAINING AS REQUIRED,INSTRUCTED THAT ALL CITY OF CHICAGO FOOD SERVICE MANAGERS MUST OBTAIN THE REQUIRED FOOD ALLERGEN TRAINING. ONE CERTIFICATE ON SITE.
2633304,10,ADEQUATE HANDWASHING SINKS PROPERLY SUPPLIED AND ACCESSIBLE,OBSERVED NO SOAP AT EXPOSED HANDWASHING SINKS LOCATED IN CENTRAL PREP AREA NEXT TO GRILL AND IN BAR. INSTRUCTED TO CORRECT AND MAINTAIN SOAP AT ALL EXPOSED HANDWASHING SINKS AT ALL TIMES. PRIORITY FOUNDATION VIOLATION 7-38-030(C). CITATION ISSUED.
2633228,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN",6-201.13 : MUST REPAIR OR REPLACE WALK IN COOLER FLOORING IN POOR REPAIR.
2633239,53,"TOILET FACILITIES: PROPERLY CONSTRUCTED, SUPPLIED, & CLEANED",OBSERVED ONE OF THE TWO TOILETS OUT OF ORDER IN THE BOYS/ADULT WASHROOM LOCATED IN THE BASEMENT. WHEN FLASHING TOILET OVERFLOWING WATER DOES NOT STOP. MUST REPAIR TOILET. PRIORITY FOUNDATION VIOLATION. 7-38-030(C). NO CITATION ISSUED.
2633239,55,"PHYSICAL FACILITIES INSTALLED, MAINTAINED & CLEAN",OBSERVED A FEW WATER DAMAGED CEILING TILES ABOVE SERVING LINE AND REAR FOOD PREP AREA. MUST REPAIR/REPLACE AND MAINTAIN FACILITY IN A GOOD REPAIR.
2633210,39,"CONTAMINATION PREVENTED DURING FOOD PREPARATION, STORAGE & DISPLAY","NOTED INDIVIDUAL TO-GO DISPOSABLE CONTAINERS AND BOWLS WITH NO HANDLE USED FOR SCOOPING FOOD AT THE SERVICE PREP COOLERS. INSTRUCTED TO PROVIDE TONGS, SERVING SPOONS, OR LADLE FOR SCOOPING FOOD."


In [0]:
df_chicago_silver = add_silver_lineage(df_chicago_clean.drop("Violations"), ["Inspection_ID", "DBA_Name", "Inspection_Date"])
(
    df_chicago_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER_SCHEMA}.chicago_inspections")
)
write_parquet_snapshot(df_chicago_silver, SILVER_PATH, "chicago_inspections")
print(f"Silver Chicago inspections: {df_chicago_silver.count()} rows")

df_chicago_violations_out = add_silver_lineage(
    df_chicago_violations.select(
        "Inspection_ID", "violation_code", "violation_description", "violation_comments", "violation_points", "source_city"
    ),
    ["Inspection_ID", "violation_code"],
)
(
    df_chicago_violations_out.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER_SCHEMA}.chicago_violations")
)
write_parquet_snapshot(df_chicago_violations_out, SILVER_PATH, "chicago_violations")
print("Silver Chicago violations table created.")

Parquet snapshot written to: /Volumes/workspace/foodinspection_silver/food_inspection/chicago_inspections/2026/04/13/run_id=20260413_181331
Silver Chicago inspections: 221877 rows
Parquet snapshot written to: /Volumes/workspace/foodinspection_silver/food_inspection/chicago_violations/2026/04/13/run_id=20260413_181331
Silver Chicago violations table created.


## 3. Dallas Silver Cleansing

In [0]:
df_dallas_clean = (
    df_dallas_bronze
    .withColumn("Restaurant_Name", trim(col("Restaurant_Name")))
    .withColumn("Street_Address", trim(col("Street_Address")))
    .withColumn("Inspection_Type", trim(col("Inspection_Type")))
    .withColumn("Zip_Code", regexp_replace(col("Zip_Code").cast("string"), "\\.0$", ""))
    .withColumn("Zip_Code", regexp_extract(col("Zip_Code"), r"^(\d{5})", 1))
    .withColumn("Zip_Code", when(col("Zip_Code") == "", lit(None)).otherwise(col("Zip_Code")))
    .withColumn("Zip_Code", when(length(col("Zip_Code")) == 4, concat_ws("", lit("0"), col("Zip_Code"))).otherwise(col("Zip_Code")))
    .withColumn("Inspection_Date", col("Inspection_Date").cast("date"))
    .withColumn("Inspection_Score", expr("try_cast(Inspection_Score as int)"))
    .withColumn("Latitude", regexp_extract(col("Lat_Long_Location"), r"\(([^,]+),", 1))
    .withColumn("Latitude", expr("try_cast(Latitude as double)"))
    .withColumn("Longitude", regexp_extract(col("Lat_Long_Location"), r",\s*([^)]+)\)", 1))
    .withColumn("Longitude", expr("try_cast(Longitude as double)"))
    .withColumn("City", lit("DALLAS"))
    .withColumn("State", lit("TX"))
    .withColumn("source_city", lit("Dallas"))
    .withColumn(
        "Results",
        when(col("Inspection_Score") >= 90, lit("Pass"))
        .when(col("Inspection_Score") >= 80, lit("Pass w/ Conditions"))
        .when(col("Inspection_Score") >= 70, lit("Fail"))
        .when(col("Inspection_Score") < 70, lit("Fail"))
        .otherwise(lit(None).cast("string")),
    )
    .withColumn("Inspection_ID", monotonically_increasing_id())
)

dallas_before = df_dallas_clean.count()

df_dallas_clean = df_dallas_clean.filter(col("Restaurant_Name").isNotNull() & (col("Restaurant_Name") != ""))
df_dallas_clean = df_dallas_clean.filter(col("Inspection_Date").isNotNull())
df_dallas_clean = df_dallas_clean.filter(col("Inspection_Type").isNotNull() & (col("Inspection_Type") != ""))
df_dallas_clean = df_dallas_clean.filter(col("Zip_Code").isNotNull() & col("Zip_Code").rlike(r"^\d{5}$"))
df_dallas_clean = df_dallas_clean.filter(
    col("Inspection_Score").isNull() | ((col("Inspection_Score") >= 0) & (col("Inspection_Score") <= 100))
)

violation_desc_cols = [c for c in df_dallas_clean.columns if c.startswith("Violation_Description")]
df_dallas_clean = df_dallas_clean.withColumn(
    "violation_count",
    sum([when(col(c).isNotNull() & (col(c) != ""), lit(1)).otherwise(lit(0)) for c in violation_desc_cols])
)

df_dallas_clean = df_dallas_clean.filter(col("violation_count") >= 1)
df_dallas_clean = df_dallas_clean.filter(~((col("Inspection_Score") >= 90) & (col("violation_count") > 3)))

dallas_after = df_dallas_clean.count()
print(f"Dallas: {dallas_before} -> {dallas_after} rows ({dallas_before - dallas_after} dropped)")

violation_structs = []
for i in range(1, 26):
    desc_col = f"Violation_Description_{i}"
    pts_col = f"Violation_Points_{i}"
    detail_col = f"Violation_Detail_{i}"
    memo_col = f"Violation_Memo_{i}"
    if desc_col in df_dallas_clean.columns:
        violation_structs.append(
            struct(
                lit(str(i)).alias("violation_num"),
                col(desc_col).alias("violation_description"),
                col(pts_col).cast(IntegerType()).alias("violation_points"),
                col(detail_col).alias("violation_detail"),
                col(memo_col).alias("violation_comments"),
            )
        )

df_dallas_violations = (
    df_dallas_clean
    .withColumn("violations_array", array(*violation_structs))
    .withColumn("violation", explode(col("violations_array")))
    .filter(col("violation.violation_description").isNotNull() & (col("violation.violation_description") != ""))
    .select(
        "Inspection_ID",
        col("violation.violation_num").alias("violation_num"),
        col("violation.violation_description").alias("violation_description_raw"),
        col("violation.violation_points").alias("violation_points"),
        col("violation.violation_detail").alias("violation_detail"),
        col("violation.violation_comments").alias("violation_comments"),
        "source_city",
    )
    .withColumn("violation_code", regexp_extract(col("violation_description_raw"), r"\*?(\d+)\s", 1))
    .withColumn("violation_code", when(col("violation_code") == "", lit(None)).otherwise(col("violation_code")))
    .withColumn("violation_description", trim(regexp_extract(col("violation_description_raw"), r"\*?\d+\s+(.*)", 1)))
    .withColumn(
        "violation_description",
        when(col("violation_description") == "", trim(col("violation_description_raw"))).otherwise(col("violation_description")),
    )
)

display(
    df_dallas_violations
    .select("Inspection_ID", "violation_code", "violation_description", "violation_points", "violation_comments")
    .limit(20)
)

Dallas: 78984 -> 54090 rows (24894 dropped)


Inspection_ID,violation_code,violation_description,violation_points,violation_comments
0,01,"Cooling -- within 2 hours, 135-70øF",3,null
0,10,"Chlorine sanitizer concentration, minimum temp.",3,null
0,31,Handwashing lavatory - accessible,2,null
0,21,RFSM - Not On Site,2,null
0,47,RFSM Certificate - Not Display,1,null
0,09,Food protected cross contamination arrange each type food in equipment so cross is prevented,3,Washisng dishes and vegetables in the the same set of sinks
0,14,When to wash hands after handling soiled equip/utensil,3,null
0,07,"Food safe, good condition, unadulterated, and honestly presented",3,Rattle snake power capsules - lack of label information
0,15,Bare hands contact with ready-to-eat foods,3,null
0,21,PIC ensures food properly and rapidly cooled,2,null


In [0]:
violation_cols_to_drop = [c for c in df_dallas_clean.columns if c.startswith("Violation_")]
violation_cols_to_drop += ["Lat_Long_Location", "Inspection_Month", "Inspection_Year", "violation_count"]
violation_cols_to_drop += ["Street_Number", "Street_Name", "Street_Direction", "Street_Type", "Street_Unit"]

df_dallas_silver = add_silver_lineage(df_dallas_clean.drop(*violation_cols_to_drop), ["Inspection_ID", "Restaurant_Name", "Inspection_Date"])
(
    df_dallas_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER_SCHEMA}.dallas_inspections")
)
write_parquet_snapshot(df_dallas_silver, SILVER_PATH, "dallas_inspections")
print(f"Silver Dallas inspections: {df_dallas_silver.count()} rows")

df_dallas_violations_out = add_silver_lineage(
    df_dallas_violations.select(
        "Inspection_ID", "violation_code", "violation_description", "violation_comments", "violation_points", "source_city"
    ),
    ["Inspection_ID", "violation_code"],
)
(
    df_dallas_violations_out.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{SILVER_SCHEMA}.dallas_violations")
)
write_parquet_snapshot(df_dallas_violations_out, SILVER_PATH, "dallas_violations")
print("Silver Dallas violations table created.")

Parquet snapshot written to: /Volumes/workspace/foodinspection_silver/food_inspection/dallas_inspections/2026/04/13/run_id=20260413_181331
Silver Dallas inspections: 54090 rows
Parquet snapshot written to: /Volumes/workspace/foodinspection_silver/food_inspection/dallas_violations/2026/04/13/run_id=20260413_181331
Silver Dallas violations table created.


In [0]:
display(spark.sql(f"""
    SELECT 'Chicago Inspections' AS table_name, COUNT(*) AS row_count FROM {SILVER_SCHEMA}.chicago_inspections
    UNION ALL
    SELECT 'Chicago Violations', COUNT(*) FROM {SILVER_SCHEMA}.chicago_violations
    UNION ALL
    SELECT 'Dallas Inspections', COUNT(*) FROM {SILVER_SCHEMA}.dallas_inspections
    UNION ALL
    SELECT 'Dallas Violations', COUNT(*) FROM {SILVER_SCHEMA}.dallas_violations
"""))

table_name,row_count
Chicago Inspections,221877
Chicago Violations,954917
Dallas Inspections,54090
Dallas Violations,308877
